# 🗣️ Bangla (Bengali) Piper TTS voice train kori — nijer YouTube video theke

Ei notebook tomar **nijer / consented** YouTube video (Bangladeshi *promito* Bangla) theke ekta **single-speaker Bangla Piper voice** train kore, ar sesh-e ekta **CPU-te chole** emon Piper voice (`.onnx` + `.onnx.json`) export kore.

**Pipeline (new Piper = [OHF-Voice/piper1-gpl](https://github.com/OHF-Voice/piper1-gpl)):**
1. YouTube audio download → 22050 Hz mono wav
2. **silero-vad** diye speech boundary te 3–15s clip banai (music/silence baad)
3. **faster-whisper large-v3** (`bn`) diye transcribe → `metadata.csv`
4. **Manual review** (khub important — whisper Bangla-te vul kore!)
5. **Finetune** kori Bangla base checkpoint (`bn_BD/google/medium`) theke
6. Export `.onnx` + test + Drive-e save

> ⚠️ **Consent:** Sudhu tomar nijer video, othoba jini permission diyechen tar video use koro. Single-speaker dhore neya hocche.

**Age koro:** Runtime → Change runtime type → **T4 GPU** → Save.

---
### 📝 Keno legacy `piper_train` na, `piper.train` (piper1-gpl)?
Purono `rhasspy/piper` repo **2025 Oct-e archive** hoye geche, ar tar `piper_train` + `piper-phonemize==1.1.0` install ekhon Colab Python 3.12-te khub **fragile**. Notun **`piper1-gpl`** maintained, `pip install -e .[train]` diye cleanly install hoy, espeak-ng directly use kore (piper-phonemize wheel jhamela nai), ei-e Bangla (`bn`) + **eki `rhasspy/piper-checkpoints`** finetune support kore. Tai amra `python -m piper.train fit` use korchi. (Legacy command gulo niche appendix-e ache.)

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU nai! Runtime > Change runtime type > T4 GPU koro.'
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

## 2. Google Drive mount koro
Training onek lomba (disconnect hote pare). Checkpoint + final voice **Drive-e** save hobe jate disconnect-eo na haray.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
DRIVE_ROOT = '/content/drive/MyDrive/piper_bangla'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)

## 3. Software install (~5–10 min)
Install kori: piper1-gpl (train stack), espeak-ng, yt-dlp, ffmpeg, faster-whisper, silero-vad, bnunicodenormalizer.
> Kono dependency **ERROR** dekhle amake dekhao — pin change korte hote pare.

In [ ]:
import os
# System deps
!apt-get -q update -y
!apt-get -q install -y build-essential cmake ninja-build espeak-ng ffmpeg aria2

# piper1-gpl (training pipeline) — pinned to a known ref for reproducibility.
# Chaile PIPER_REF badle newest paite paro, kintu fragile hole ekta release tag boshao.
PIPER_REF = 'main'   # e.g. 'v1.3.0' — install/train fail korle ekta stable tag boshao
REPO_DIR = '/content/piper1-gpl'
if not os.path.exists(REPO_DIR):
    !git clone -q -b {PIPER_REF} https://github.com/OHF-Voice/piper1-gpl.git {REPO_DIR}
%cd {REPO_DIR}
!python -m pip install -q -e '.[train]'
!bash build_monotonic_align.sh
!python setup.py build_ext --inplace
%cd /content

# Data / preprocessing tools
!pip install -q yt-dlp faster-whisper silero-vad soundfile bnunicodenormalizer huggingface_hub
print('\n✅ Install done. (upore kono ERROR thakle amake bolo)')

## 4. Input — tomar YouTube link + model name
`YT_LINKS` e tomar **nijer/consented** video-r link poste koro. Beshi variety = valo (kintu **eki speaker** hote hobe).

**Data koto lagbe?** Finetune-e minimum **~20–30 min** clean speech (moto boshano jai), **1–2 ghonta** hole valo, **3+ ghonta** hole best.

In [ ]:
YT_LINKS = [
    # "https://www.youtube.com/watch?v=XXXXXXXXXXX",
    # "https://www.youtube.com/watch?v=YYYYYYYYYYY",
]
MODEL_NAME = 'bn_bd-myvoice'   # voice-r nam (file naming-e use hobe)

SAMPLE_RATE = 22050            # Piper medium = 22050 (change korio na)
ESPEAK_VOICE = 'bn'            # espeak-ng Bengali code

# Working dirs
import os
DATASET_DIR = '/content/dataset'
WAV_DIR     = os.path.join(DATASET_DIR, 'wavs')
META_CSV    = os.path.join(DATASET_DIR, 'metadata.csv')
RAW_DIR     = '/content/raw_audio'
for d in (WAV_DIR, RAW_DIR):
    os.makedirs(d, exist_ok=True)
assert YT_LINKS, '⚠️ YT_LINKS khali! Age tomar link boshao.'
print(f'{len(YT_LINKS)} ta link, model =', MODEL_NAME)

## 5. Prottek video-r audio download → 22050 Hz mono wav
yt-dlp bestaudio nibe, ffmpeg diye 22050 Hz mono wav banabe.

In [ ]:
import glob, subprocess, os
raw_wavs = []
for i, url in enumerate(YT_LINKS):
    src = f'{RAW_DIR}/src_{i}.%(ext)s'
    print(f'[{i+1}/{len(YT_LINKS)}] downloading:', url)
    !python -m yt_dlp -f bestaudio/best --no-playlist --quiet --no-warnings -x -o "{src}" "{url}"
    got = sorted(glob.glob(f'{RAW_DIR}/src_{i}.*'))
    if not got:
        print('   ⚠️ download fail:', url); continue
    out = f'{RAW_DIR}/full_{i}.wav'
    subprocess.run(['ffmpeg','-y','-i',got[0],'-ac','1','-ar',str(SAMPLE_RATE),out,'-loglevel','error'])
    raw_wavs.append(out)
    print('   ✓', out)
print('\nTotal full recordings:', len(raw_wavs))
assert raw_wavs, 'Kono audio download hoy nai — link check koro.'

## 6. silero-vad diye 3–15s clip-e kata
Speech boundary te kete, khub choto/music/silence clip baad debe. Prottek clip 22050 Hz mono wav hisebe `dataset/wavs/`-e jabe.

In [ ]:
import torch, soundfile as sf, numpy as np, os, glob
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps

vad = load_silero_vad()
MIN_S, MAX_S = 3.0, 15.0     # clip length range (Piper: 3-15s valo)
clip_id = 0
# purono clip muche felo
for f in glob.glob(WAV_DIR + '/*.wav'):
    os.remove(f)

for wav_path in raw_wavs:
    wav = read_audio(wav_path, sampling_rate=SAMPLE_RATE)
    ts = get_speech_timestamps(
        wav, vad, sampling_rate=SAMPLE_RATE,
        min_speech_duration_ms=int(MIN_S*1000),
        max_speech_duration_s=MAX_S,
        min_silence_duration_ms=300,
        speech_pad_ms=200,
    )
    full = wav.numpy()
    for seg in ts:
        a, b = seg['start'], seg['end']
        dur = (b - a) / SAMPLE_RATE
        if dur < MIN_S or dur > MAX_S:
            continue
        clip = full[a:b]
        # RMS check — khub soft/empty clip baad
        if np.sqrt(np.mean(clip**2)) < 0.005:
            continue
        clip_id += 1
        sf.write(f'{WAV_DIR}/{clip_id}.wav', clip, SAMPLE_RATE, subtype='PCM_16')

print(f'✅ {clip_id} ta clip banano holo → {WAV_DIR}')
tot = sum(sf.info(f).duration for f in glob.glob(WAV_DIR+'/*.wav'))
print(f'Total usable speech: {tot/60:.1f} min')
assert clip_id > 0, 'Kono clip banano jai nai — audio/VAD setting check koro.'

## 7. faster-whisper (`bn`) diye transcribe → metadata.csv
`large-v3`, language `bn`. Bangla text `bnunicodenormalizer` diye normalize kori.
Format: `clipid.wav|text` (Piper single-speaker — 2 column, `|` delimiter, UTF-8).

In [ ]:
from faster_whisper import WhisperModel
from bnunicodenormalizer import Normalizer
import glob, os, torch

bnorm = Normalizer()
def normalize_bn(text):
    out = []
    for w in text.strip().split():
        try:
            r = bnorm(w)
            out.append(r['normalized'] if r and r.get('normalized') else w)
        except Exception:
            out.append(w)
    return ' '.join(out).strip()

wm = WhisperModel('large-v3', device='cuda', compute_type='float16')
rows = []
clips = sorted(glob.glob(WAV_DIR+'/*.wav'), key=lambda p:int(os.path.splitext(os.path.basename(p))[0]))
for i, c in enumerate(clips):
    segs, _ = wm.transcribe(c, language='bn', beam_size=5)
    text = ''.join(s.text for s in segs).strip().replace('\n', ' ')
    text = normalize_bn(text)
    if not text:
        continue
    rows.append(f'{os.path.basename(c)}|{text}')
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(clips)} done')

with open(META_CSV, 'w', encoding='utf-8') as f:
    f.write('\n'.join(rows) + '\n')

del wm; import gc; gc.collect(); torch.cuda.empty_cache()
print(f'\n✅ metadata.csv likha holo: {len(rows)} ta line → {META_CSV}')
print('\nProthom 5 line:')
print('\n'.join(rows[:5]))

## 8. ⚠️⚠️ MANUAL REVIEW — metadata.csv thik koro (SOBCHEYE IMPORTANT step)

**Transcript-er accuracy = voice quality-r #1 factor.** Whisper Bangla-te niyomito vul kore (matra, juktokkhor, punctuation, English word). Ekhane time dile voice onek valo hobe.

**Ki korbe:**
1. Niche `metadata.csv` download koro (othoba Colab file browser theke edit koro).
2. Prottek line-er text **shune-shune** thik koro (clip `dataset/wavs/<id>.wav`).
3. Kharap/oshpokto clip-er line **muche felo** (clip-o baad porbe).
4. Thik kora file abar upload koro (**next cell**).

> Tumi chaile ei step skip korte paro, kintu quality tulnamulok kharap hobe.

In [ ]:
# metadata.csv download kore hate thik koro:
from google.colab import files
files.download(META_CSV)
print('Download holo. Thik kore niche-r cell diye abar upload koro (na korle ei-tai use hobe).')

In [ ]:
# Thik kora metadata.csv abar upload koro (optional):
from google.colab import files
import shutil, os
up = files.upload()   # thik kora file select koro (nam metadata.csv na holeo cholbe)
if up:
    src = list(up.keys())[0]
    shutil.move(src, META_CSV)
    print('✅ Updated metadata.csv boshano holo.')
# validation
n = sum(1 for _ in open(META_CSV, encoding='utf-8') if _.strip())
print(f'metadata.csv e ekhon {n} ta line ache.')

## 9. Bangla base checkpoint download (finetune-er jonno)
`rhasspy/piper-checkpoints` e **Bangla (`bn_BD/google/medium`)** checkpoint ache (22050 Hz, medium, espeak `bn`) — eta LibriTTS-R theke finetuned, **16 speaker** (multi-speaker).

Amra single-speaker train korbo, tai `--ckpt_path` (strict) kaj korbe na (speaker-embedding shape mismatch). Er bodole **`--model.warmstart_ckpt`** use korbo — eta **non-strict**: shob matching-shape param (Bangla phoneme embedding + encoder + decoder + vocoder) copy kore, sudhu speaker-embedding (16→1) skip kore. Erfole choto data-teo Bangla quality onek valo hoy.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

BASE_DIR = '/content/base_ckpt'
os.makedirs(BASE_DIR, exist_ok=True)
repo = 'rhasspy/piper-checkpoints'
ck = hf_hub_download(repo, 'bn/bn_BD/google/medium/last.ckpt', repo_type='dataset')
cfg = hf_hub_download(repo, 'bn/bn_BD/google/medium/config.json', repo_type='dataset')
BASE_CKPT = os.path.join(BASE_DIR, 'bn_base.ckpt')
shutil.copy(ck, BASE_CKPT)
print('✅ Base Bangla checkpoint:', BASE_CKPT)
print('   size:', round(os.path.getsize(BASE_CKPT)/1e6, 1), 'MB')

## 10. Train (finetune) — checkpoint Drive-e save hobe

- **Output/checkpoint** `--trainer.default_root_dir` → **Drive** (disconnect-e survive kore).
- **Precision** `16-mixed` (T4-te fast).
- **Batch size** 12 (T4 16GB medium VITS-e nirapod; OOM hole 8 koro).
- **max_epochs** boro rakha — **je-kono somoy stop kore** sesh checkpoint use korte paro; TensorBoard-e sample shune bujhbe kobe valo.

**T4-te time:** ~30–60 min-e prothom bojhar moto output; **valo quality-r jonno koyek ghonta** (finetune bole scratch-er cheye onek druto). Disconnect hole niche-r **Resume** cell chalaao.

In [ ]:
import os
OUT_DIR   = os.path.join(DRIVE_ROOT, MODEL_NAME)      # checkpoints/logs Drive-e
CACHE_DIR = '/content/cache'                          # phoneme/audio cache (local ok)
CONFIG_OUT= os.path.join(OUT_DIR, 'config.json')      # training-e likha hobe (pore .onnx.json)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

BATCH_SIZE = 12
MAX_EPOCHS = 4000

# Ekbar finetune shuru: warmstart theke. (Resume-er jonno niche-r cell.)
cmd = f'''cd {REPO_DIR} && python -m piper.train fit \
  --data.voice_name "{MODEL_NAME}" \
  --data.csv_path "{META_CSV}" \
  --data.audio_dir "{WAV_DIR}" \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "{CACHE_DIR}" \
  --data.config_path "{CONFIG_OUT}" \
  --data.batch_size {BATCH_SIZE} \
  --model.sample_rate {SAMPLE_RATE} \
  --model.warmstart_ckpt "{BASE_CKPT}" \
  --trainer.default_root_dir "{OUT_DIR}" \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.precision 16-mixed \
  --trainer.log_every_n_steps 20'''
print(cmd, '\n')
get_ipython().system(cmd)

### 🔁 Resume (disconnect holo? ei cell chalaao)
Warmstart **na** kore amader nijer sesh `last.ckpt` theke strict resume kore (same architecture). Upore-r train cell abar cholabe **na** — sudhu eita chalaao.

In [ ]:
import glob, os, re
pat = os.path.join(OUT_DIR, 'lightning_logs', '**', 'checkpoints', 'last.ckpt')
cks = glob.glob(pat, recursive=True)
assert cks, 'Kono last.ckpt paoa jai nai — age ekbar train cell chalaao.'
def vnum(p):
    m = re.search(r'version_(\d+)', p); return int(m.group(1)) if m else -1
RESUME = sorted(cks, key=vnum)[-1]
print('Resuming from:', RESUME)
cmd = f'''cd {REPO_DIR} && python -m piper.train fit \
  --data.voice_name "{MODEL_NAME}" \
  --data.csv_path "{META_CSV}" \
  --data.audio_dir "{WAV_DIR}" \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "{CACHE_DIR}" \
  --data.config_path "{CONFIG_OUT}" \
  --data.batch_size {BATCH_SIZE} \
  --model.sample_rate {SAMPLE_RATE} \
  --trainer.default_root_dir "{OUT_DIR}" \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.precision 16-mixed \
  --trainer.log_every_n_steps 20 \
  --ckpt_path "{RESUME}"'''
print(cmd, '\n')
get_ipython().system(cmd)

## 11. Export → CPU-runnable Piper voice (`.onnx` + `.onnx.json`)
Sesh checkpoint (`last.ckpt` othoba TensorBoard-e best-shona ekta) export kore, ar training-er `config.json`-ke `.onnx.json` naame copy kori.

In [ ]:
import glob, os, re, shutil
pat = os.path.join(OUT_DIR, 'lightning_logs', '**', 'checkpoints', '*.ckpt')
cks = glob.glob(pat, recursive=True)
assert cks, 'Kono checkpoint nai — age train koro.'
# default: newest last.ckpt; chaile EXPORT_CKPT hate boshao
lasts = [c for c in cks if c.endswith('last.ckpt')]
EXPORT_CKPT = sorted(lasts or cks, key=os.path.getmtime)[-1]
print('Exporting:', EXPORT_CKPT)

VOICE_BASE = f'{MODEL_NAME}-medium'
ONNX_OUT   = f'/content/{VOICE_BASE}.onnx'
!cd {REPO_DIR} && python -m piper.train.export_onnx --checkpoint "{EXPORT_CKPT}" --output-file "{ONNX_OUT}"

# config.json (training-e likha) → .onnx.json
shutil.copy(CONFIG_OUT, ONNX_OUT + '.json')
print('✅ Voice ready:')
print('  ', ONNX_OUT)
print('  ', ONNX_OUT + '.json')

## 12. Test — promito Bangla sentence bolao, shono
Piper Python API diye CPU-te synth kore wav banai.

In [ ]:
import wave
from piper import PiperVoice
from IPython.display import Audio, display

voice = PiperVoice.load(ONNX_OUT)   # CPU-te chole (use_cuda=True dile GPU)
TEST_TEXT = 'সুপ্রভাত। আজ আবহাওয়া চমৎকার। আপনি কেমন আছেন?'
TEST_WAV = '/content/test_bangla.wav'
with wave.open(TEST_WAV, 'wb') as wf:
    voice.synthesize_wav(TEST_TEXT, wf)
print('Text:', TEST_TEXT)
display(Audio(TEST_WAV))

## 13. Final voice Drive-e save + download

In [ ]:
import shutil, os
from google.colab import files
FINAL_DIR = os.path.join(DRIVE_ROOT, 'final_voice')
os.makedirs(FINAL_DIR, exist_ok=True)
for f in (ONNX_OUT, ONNX_OUT + '.json'):
    shutil.copy(f, FINAL_DIR)
    print('Saved to Drive:', os.path.join(FINAL_DIR, os.path.basename(f)))
# browser-eo download:
files.download(ONNX_OUT)
files.download(ONNX_OUT + '.json')
print('\n✅ Done! Ei .onnx + .onnx.json diye Piper CPU-te Bangla bolbe.')

---
## 📌 TODO / Verify korar bishoy (research theke)

- **piper1-gpl version pin:** upore `PIPER_REF='main'` deya. Kokhono `main`-e breaking change ashte pare — install/train fail korle repo-r [Releases](https://github.com/OHF-Voice/piper1-gpl/releases) theke ekta stable **tag** (jemon `v1.3.0`) boshao.
- **Colab Python 3.12 / torch:** `pip install -e '.[train]'` nijer torch/lightning pin ane. Jodi PyTorch 2.6+ `weights_only` niye checkpoint-load error dei, warmstart/export code-e `torch.load(..., weights_only=False)` already ache; tobuo `--ckpt_path` resume-e error hole ei point-ta verify koro.
- **warmstart vs full quality:** warmstart Bangla checkpoint-er speaker-embedding (16→1) skip kore. Eta expected — baki shob Bangla weight copy hoy. Full multi-speaker rakhte chaile alada setup lagbe (ei notebook single-speaker).
- **espeak `bn`:** ek-i `bn` code Bangladesh + India Bengali-r jonno (espeak-e bn_BD alada nai). Promito-r jonno thik ache.
- **max_epochs:** 4000 arbitrary upper bound. Realistically TensorBoard-e sample shune valo lagle stop koro; finetune-e onek age-i decent hoy.

## 📓 Appendix: legacy `rhasspy/piper` (`piper_train`) — sudhu reference
Purono repo archived + Colab-e fragile bole amra upore piper1-gpl use korechi. Kintu keu purono pipeline chaile (known-good pin, [discussion #736](https://github.com/rhasspy/piper/discussions/736)):
```
pip install cython>=0.29.0 piper-phonemize==1.1.0 librosa>=0.9.2 numpy==1.26.4 \
    onnxruntime>=1.11.0 pytorch-lightning==1.9.0 torch==2.0.0 torchaudio==2.0.0 \
    torchmetrics==0.11.4 --extra-index-url https://download.pytorch.org/whl/cu117
# preprocess (LJSpeech, single-speaker, id|text):
python -m piper_train.preprocess --language bn --input-dir DATA --output-dir TRAIN \
    --dataset-format ljspeech --single-speaker --sample-rate 22050
# train (finetune):
python -m piper_train --dataset-dir TRAIN --accelerator gpu --devices 1 --batch-size 16 \
    --validation-split 0.0 --num-test-examples 0 --max_epochs 10000 \
    --checkpoint-epochs 1 --precision 32 --resume_from_checkpoint bn_base.ckpt
# export:
python -m piper_train.export_onnx model.ckpt model.onnx
cp TRAIN/config.json model.onnx.json
```
> Sotorko: `piper-phonemize==1.1.0`-er Colab Python 3.12 wheel na thakle ei path bhaggchore. Tai piper1-gpl-i recommended.